<a href="https://colab.research.google.com/github/alter-mix-dev/HACKATON2/blob/feat%2Fpruebas/bot_cocina.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Leer credenciales desde Colab Secrets

from google.colab import userdata

VERIFY_TOKEN = "mi_token"
WHATSAPP_TOKEN = userdata.get('WHATSAPP_ACCESS_TOKEN')
PHONE_ID = userdata.get('PHONE_NUMBER_ID')

print("Credenciales cargadas correctamente.")

Credenciales cargadas correctamente.


In [2]:
# Instalar las librerías necesarias en la máquina virtual de Colab
!pip install pyngrok fastapi uvicorn llama-stack-client pandas datasets --quiet

import os
import time
import yaml
import json
import re
import threading
import unicodedata
import subprocess
import requests
import pandas as pd
from fastapi import FastAPI, Request, BackgroundTasks
import uvicorn
from pyngrok import ngrok
from google.colab import userdata
from llama_stack_client import LlamaStackClient, APITimeoutError, APIConnectionError, APIStatusError
from datasets import load_dataset
print("✅ Dependencias instaladas con éxito.")

✅ Dependencias instaladas con éxito.


In [3]:

_ = '''
opcion 2 a refinar en un futuro. añadí datasets y from datasets import load_dataset solo para trabajar con este .cvs
la url es https://huggingface.co/datasets/somosnlp/RecetasDeLaAbuela/blob/main/recetasdelaabuela.csv
print("📥 Descargando y estructurando catálogo de recetas (44.5 MB)...")
try:
    # 1. Cargamos el archivo CSV directamente apuntando al repositorio de Hugging Face
    # La librería 'datasets' detecta el formato y optimiza el consumo de memoria en Colab
    dataset = load_dataset(
        "somosnlp/RecetasDeLaAbuela",
        data_files="recetasdelaabuela.csv"
    )

    # 2. Convertimos el conjunto de datos de entrenamiento ('train') a un DataFrame de Pandas
    df_catalogo = dataset['train'].to_pandas()

    print(f"✅ Catálogo cargado con éxito. {len(df_catalogo)} recetas enriquecidas disponibles.")

    # 3. Vista rápida de la estructura interna para la lógica de tu bot
    print(f"📋 Metadatos del recetario listos para usar: {list(df_catalogo.columns)}")

except Exception as e:
    print(f"❌ Error crítico al inicializar el dataset de cocina: {e}")
    df_catalogo = pd.DataFrame()
######## opcion 1#############
#_ = '''
# CONFIGURACION DEL RECETARIO IMPORTANDO EL DATASET COMPRIMIDO CON 173 000 RECETAS
# 1. CONFIGURACIÓN DEL ENTORNO
import io
import gzip
import urllib.request
#import pandas as pd
# importacion y configuracion del dataset
JSON_URL = "https://raw.githubusercontent.com/jakevdp/open-recipe-data/main/recipeitems.json.gz"

print("📥 Cargando catálogo masivo de recetas de cocina en memoria...")
try:
    # 1. Agregamos un User-Agent para que GitHub no bloquee o altere la descarga
    req = urllib.request.Request(
        JSON_URL,
        headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
    )

    # 2. Descargamos el archivo comprimido en un flujo de bytes
    with urllib.request.urlopen(req) as response:
        compressed_file = io.BytesIO(response.read())

        # 3. Descomprimimos el contenido .gz en tiempo real
        with gzip.open(compressed_file, 'rb') as f:
            # 4. Cargamos el JSON por líneas en Pandas
            df_catalogo = pd.read_json(f, lines=True)

    print(f"✅ Catálogo cargado con éxito. {len(df_catalogo)} recetas disponibles.")

except Exception as e:
    print(f"❌ Error crítico al cargar el dataset: {e}")
    df_catalogo = pd.DataFrame()
    #'''

📥 Cargando catálogo masivo de recetas de cocina en memoria...
✅ Catálogo cargado con éxito. 173278 recetas disponibles.


In [4]:
# 1.1 Instalar Ollama

!apt-get update -qq
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.5.5+dfsg2-2build1.1).
0 upgraded, 0 newly installed, 0 to remove and 54 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [5]:
# 1.2 Iniciar el servidor de Ollama en segundo plano

import subprocess
import time

proceso_ollama = subprocess.Popen(["ollama", "serve"])
time.sleep(5)

!pip install ollama --quiet
import ollama

!ollama pull llama3.1:8b
print("Ollama listo.")


Ollama listo.


In [6]:
#INSTALAR Y LEVANTAR LLAMA STACK
!pip install "llama-stack[starter]" llama-stack-client --quiet
!pip install "mcp<2.0" --quiet

In [7]:
# Instalar Llama Stack (servidor + cliente) y levantarlo con Ollama registrado

import subprocess
import time
import os
import yaml
import llama_stack

os.environ["OLLAMA_URL"] = "http://localhost:11434/v1"

# Generar un config de Llama Stack con el modelo de Ollama ya registrado
config_path = os.path.join(os.path.dirname(llama_stack.__file__), "distributions", "starter", "config.yaml")
with open(config_path) as f:
    config = yaml.safe_load(f)

config["registered_resources"]["models"].append({
    "model_id": "llama3.1:8b",
    "provider_id": "ollama",
    "provider_model_id": "llama3.1:8b",
    "model_type": "llm",
})

mi_config_path = "/content/mi_config_llama_stack.yaml"
with open(mi_config_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

# Levantar el servidor con ese config
proceso_llama_stack = subprocess.Popen(
    ["llama", "stack", "run", mi_config_path, "--port", "8321"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
time.sleep(20)  # tarda mas en levantar que Ollama

print("Proceso vivo:", proceso_llama_stack.poll() is None)
if proceso_llama_stack.poll() is not None:
    print("--- Salida del proceso ---")
    print(proceso_llama_stack.stdout.read())
else:
    from llama_stack_client import LlamaStackClient
    client = LlamaStackClient(base_url="http://localhost:8321")
    print("Llama Stack listo.")

Proceso vivo: True
Llama Stack listo.


In [8]:
# Configurar el servidor del webhook

from fastapi import FastAPI, Request, BackgroundTasks
import requests
import ollama

app = FastAPI()

mensajes_recibidos = []
usuarios_conocidos = set()

In [9]:
# Definir función de normalización de número

def normalizar_numero_mx(numero):
    if numero.startswith("521") and len(numero) == 13:
        return "52" + numero[3:]
    return numero

In [10]:
# ==========================================
# 2. DEFINICIÓN DE HERRAMIENTAS (FUNCTION CALLING)
# ==========================================

def buscar_receta_cocina(criterio: str):
    """
    Busca recetas en el catálogo en memoria por nombre o descripción y devuelve tarjetas estilizadas.
    """
    if df_catalogo.empty:
        return "⚠️ Lo siento, nuestro recetario digital no está disponible en este momento."

    criterio_norm = criterio.lower().strip()

    # Búsqueda de coincidencias parciales en el nombre o la descripción de la receta
    mascara = df_catalogo['name'].str.lower().str.contains(criterio_norm) | df_catalogo['description'].str.lower().str.contains(criterio_norm)
    resultados = df_catalogo[mascara]

    if resultados.empty:
        return f"🔎 No encontré recetas que coincidan con '{criterio}'. ¿Te gustaría intentar buscando con otro ingrediente o platillo?"

    lista_recetas = []
    # Limitamos a los 3 primeros resultados para no saturar la pantalla de WhatsApp
    for _, fila in resultados.head(3).iterrows():
        nombre = fila.get('name', 'Receta sin nombre')
        descripcion = fila.get('description', 'Sin descripción disponible.')

        # Formateo visual avanzado con estructura limpia para WhatsApp
        ficha = (
            f"🍳 *{nombre.upper()}*\\n"
            f" └ 📝 *Preparación:* {descripcion}"
        )
        lista_recetas.append(ficha)

    return "\\n\\n---\\n\\n".join(lista_recetas)

# Esquema técnico de la función para el catálogo de Llama Stack
HERRAMIENTA_BUSQUEDA = {
    "type": "function",
    "name": "buscar_receta_cocina",
    "description": "Busca recetas de cocina disponibles en el catálogo. Úsala cuando el usuario pregunte por cómo preparar un platillo o qué cocinar con ciertos ingredientes.",
    "parameters": {
        "type": "object",
        "properties": {
            "criterio": {
                "type": "string",
                "description": "El ingrediente o nombre del platillo a buscar (ej. 'patatas', 'pollo', 'tortilla')."
            }
        },
        "required": ["criterio"]
    }
}

HERRAMIENTAS = [HERRAMIENTA_BUSQUEDA]
FUNCIONES_DISPONIBLES = {
    "buscar_receta_cocina": buscar_receta_cocina,
}

In [11]:
# ==========================================
# 3. LÓGICA DE NEGOCIO Y MODELO DE IA
# ==========================================

VERIFY_TOKEN = "mi_token"
WHATSAPP_TOKEN = userdata.get('WHATSAPP_ACCESS_TOKEN')
PHONE_ID = userdata.get('PHONE_NUMBER_ID')

client = LlamaStackClient(base_url="http://localhost:8321", timeout=300)

SYSTEM_PROMPT = (
    "Eres un chef profesional y un asistente de cocina muy amable. "
    "REGLA DE ORO: Cuando el usuario te solicite ideas o te proporcione un ingrediente, proporciona una lista de 5 recetas resumidas, excepto si el usuario te pide un numero determinado de recetas"
    "REGLA DE ORO:Cuando el usuario lo solicite o elija una receta en particular, proporciona la receta completa con lista de ingredientes e instrucciones "
    "REGLA DE ORO. siempre numera las recetas"
    "Tu objetivo es guiar al usuario y resolver sus dudas sobre cómo preparar las recetas del catálogo. "
    "Responde siempre en español, de forma muy empática, clara y concisa (máximo 3-4 líneas por mensaje). "
    "Usa obligatoriamente la herramienta 'buscar_receta_cocina' cuando te soliciten recetas, te den algun ingrediente, ideas de comida o preparaciones. "
    #"Cuando la herramienta te devuelva las instrucciones con sus conectores (└), MANTÉN esa misma "
    #"estructura visual organizada en tu respuesta final de WhatsApp. "
    "Básate únicamente en los datos de recetas devueltos por la herramienta. Si un ingrediente no existe, invita al usuario a buscar otra opción. "
    "Nunca menciones nombres de funciones técnicas ni variables de código en la conversación."

)

conversaciones_por_numero = {}

def obtener_conversacion(numero):
    if numero not in conversaciones_por_numero:
        conversacion = client.conversations.create()
        conversaciones_por_numero[numero] = conversacion.id
    return conversaciones_por_numero[numero]

def generar_respuesta(numero, texto_usuario):
    conversation_id = obtener_conversacion(numero)
    respuesta = client.responses.create(
        model="ollama/llama3.1:8b",
        input=texto_usuario,
        conversation=conversation_id,
        instructions=SYSTEM_PROMPT,
        tools=HERRAMIENTAS,
    )

    llamadas_funcion = [item for item in respuesta.output if item.type == "function_call"]
    if llamadas_funcion:
        resultados = []
        for llamada in llamadas_funcion:
            funcion = FUNCIONES_DISPONIBLES[llamada.name]
            argumentos = json.loads(llamada.arguments)
            resultado = funcion(**argumentos)
            resultados.append({
                "type": "function_call_output",
                "call_id": llamada.call_id,
                "output": resultado,
            })
        respuesta = client.responses.create(
            model="ollama/llama3.1:8b",
            input=resultados,
            conversation=conversation_id,
            instructions=SYSTEM_PROMPT,
            tools=HERRAMIENTAS,
        )

    return respuesta.output_text

In [12]:
# DESPLIEGUE DEL SERVIDOR. DEFINIR LOS END POINTS

In [13]:
# ==========================================
# 4. INFRAESTRUCTURA DEL WEBHOOK (FASTAPI)
# ==========================================

app = FastAPI()
mensajes_recibidos = []
ids_procesados = set()
respuestas_enviadas = []
errores_procesamiento = []
locks_por_numero = {}

def normalizar_numero_mx(numero):
    if numero.startswith("521") and len(numero) == 13:
        return "52" + numero[3:]
    return numero

def obtener_lock(numero):
    if numero not in locks_por_numero:
        locks_por_numero[numero] = threading.Lock()
    return locks_por_numero[numero]

@app.get("/webhook")
async def verificar_webhook(request: Request):
    params = request.query_params
    if params.get("hub.verify_token") == VERIFY_TOKEN:
        return int(params.get("hub.challenge"))
    return {"error": "token invalido"}

@app.post("/webhook")
async def recibir_mensaje(request: Request, background_tasks: BackgroundTasks):
    body = await request.json()
    mensajes_recibidos.append(body)
    try:
        mensaje = body["entry"][0]["changes"][0]["value"]["messages"][0]
        wamid = mensaje["id"]
        if wamid in ids_procesados:
            return {"status": "ok"}
        ids_procesados.add(wamid)
        numero = normalizar_numero_mx(mensaje["from"])
        texto = mensaje["text"]["body"]
        background_tasks.add_task(procesar_y_responder, numero, texto)
    except (KeyError, IndexError):
        pass
    return {"status": "ok"}
'''
def enviar_respuesta(numero, texto):
    url = f"https://facebook.com{PHONE_ID}/messages"
    headers = {"Authorization": f"Bearer {WHATSAPP_TOKEN}", "Content-Type": "application/json"}
    payload = {
        "messaging_product": "whatsapp",
        "to": numero,
        "type": "text",
        "text": {"body": texto}
    }
    r = requests.post(url, headers=headers, json=payload)
    respuestas_enviadas.append({"numero": numero, "status": r.status_code, "texto": r.text})
    '''
    # Definir funcion de envio de respuesta

#respuestas_enviadas = []

def enviar_respuesta(numero, texto):
    url = f"https://graph.facebook.com/v25.0/{PHONE_ID}/messages"
    headers = {"Authorization": f"Bearer {WHATSAPP_TOKEN}", "Content-Type": "application/json"}
    payload = {
        "messaging_product": "whatsapp",
        "to": numero,
        "type": "text",
        "text": {"body": texto}
    }

    r = requests.post(url, headers=headers, json=payload)
    respuestas_enviadas.append({"numero": numero, "status": r.status_code, "texto": r.text})

In [14]:
# ==========================================
# 5. REGLAS FIJAS Y CONTROL DE FLUJO
# ==========================================

FRASES_URGENCIA = ["quemado", "emergencia", "fuego", "corte", "auxilio"]
FRASES_ESCALAMIENTO = ["hablar con humano", "chef real", "persona", "agente", "humano", "queja"]
FRASES_SALUDO = ["hola", "buenas", "buenos dias", "buenos días", "buenas tardes", "buenas noches", "que tal", "buen dia"]

def es_solo_saludo(texto_low):
    texto_limpio = texto_low.strip(" ,.!¡¿?")
    return texto_limpio in FRASES_SALUDO

def enviar_aviso_de_error(numero):
    try:
        enviar_respuesta(numero, "🤖 Ups, estoy experimentando una pequeña desconexión técnica en mi cocina. Por favor, intenta de nuevo.")
    except Exception:
        pass

def procesar_y_responder(numero, texto):
    with obtener_lock(numero):
        try:
            texto_low = texto.lower()
            if any(frase in texto_low for frase in FRASES_URGENCIA):
                enviar_respuesta(numero, "🚨 Si hay una emergencia real en la cocina (fuego o accidente), por favor aparta el teléfono y llama a los servicios de emergencia de tu país de inmediato.")
            elif any(frase in texto_low for frase in FRASES_ESCALAMIENTO):
                enviar_respuesta(numero, "✨ ¡Entendido! En un momento te transferiré con un Chef de nuestro equipo humano para asesoría culinaria personalizada.")
            elif es_solo_saludo(texto_low):
                enviar_respuesta(numero, "🍳 ¡Hola! Soy tu asistente de cocina virtual. Dime qué ingredientes tienes a la mano o qué receta te gustaría preparar hoy y buscaré en el recetario.")
            else:
                respuesta = generar_respuesta(numero, texto)
                enviar_respuesta(numero, respuesta)
        except APITimeoutError as e:
            errores_procesamiento.append({"numero": numero, "texto": texto, "causa": "llama_timeout", "error": repr(e)})
            enviar_aviso_de_error(numero)
        except APIConnectionError as e:
            errores_procesamiento.append({"numero": numero, "texto": texto, "causa": "llama_stack_desconectado", "error": repr(e)})
            enviar_aviso_de_error(numero)
        except APIStatusError as e:
            errores_procesamiento.append({"numero": numero, "texto": texto, "causa": "error_http_stack", "error": repr(e)})
            enviar_aviso_de_error(numero)
        except KeyError as e:
            errores_procesamiento.append({"numero": numero, "texto": texto, "causa": "herramienta_invalida", "error": repr(e)})
            enviar_aviso_de_error(numero)
        except requests.exceptions.RequestException as e:
            errores_procesamiento.append({"numero": numero, "texto": texto, "causa": "error_api_whatsapp", "error": repr(e)})
        except Exception as e:
            errores_procesamiento.append({"numero": numero, "texto": texto, "causa": "error_indeterminado", "error": repr(e)})
            enviar_aviso_de_error(numero)

DEFINIR LOS ENDPOINTS DEL WEBHOOK
Se definen los dos endpoints que WhatsApp necesita: uno para la verificación inicial del webhook, y otro para recibir cada mensaje entrante. La recepción delega el trabajo pesado a BackgroundTasks, para responderle a Meta de inmediato sin esperar a que Llama genere la respuesta — evitando así que Meta reintente la entrega y se dupliquen las respuestas.

In [15]:
_ = '''
# Definir endpoint de verificacion de credenciales

@app.get("/webhook")
async def verificar_webhook(request: Request):
    params = request.query_params

    if params.get("hub.verify_token") == VERIFY_TOKEN:
        return int(params.get("hub.challenge"))

    return {"error": "token invalido"}
    '''

In [16]:
_ = '''
# Definir endpoint de recepcion de mensajes

@app.post("/webhook")
async def recibir_mensaje(request: Request, background_tasks: BackgroundTasks):
    body = await request.json()
    mensajes_recibidos.append(body)

    try:
        mensaje = body["entry"][0]["changes"][0]["value"]["messages"][0]
        numero = normalizar_numero_mx(mensaje["from"])
        texto = mensaje["text"]["body"]
        background_tasks.add_task(procesar_y_responder, numero, texto)
    except (KeyError, IndexError):
        pass

    return {"status": "ok"}
    '''

In [17]:
#_ = '''
# Definir funcion de envio de respuesta

respuestas_enviadas = []

def enviar_respuesta(numero, texto):
    url = f"https://graph.facebook.com/v25.0/{PHONE_ID}/messages"
    headers = {"Authorization": f"Bearer {WHATSAPP_TOKEN}", "Content-Type": "application/json"}
    payload = {
        "messaging_product": "whatsapp",
        "to": numero,
        "type": "text",
        "text": {"body": texto}
    }

    r = requests.post(url, headers=headers, json=payload)
    respuestas_enviadas.append({"numero": numero, "status": r.status_code, "texto": r.text})
    #'''

In [18]:
# ==========================================
# 6. INICIALIZACIÓN DEL TÚNEL (NGROK)
# ==========================================
try:
    ngrok.set_auth_token(userdata.get('NGROK_AUTHTOKEN'))
    # Cerramos conexiones huérfanas antes de levantar
    tunnels = ngrok.get_tunnels()
    for tunnel in tunnels:
        ngrok.disconnect(tunnel.public_url)
    ngrok.kill()
    time.sleep(2)
except Exception:
    pass

tunel_publico = ngrok.connect(8000, domain="mutate-stylus-facedown.ngrok-free.dev")
print("\n🚀 CONFIGURACIÓN DE BOT DE COCINA EXITOSA 🚀")
print("==================================================================")
print("Configura esta URL exacta como tu Callback URL en Meta Dashboard:")
print(f"{tunel_publico.public_url}/webhook")
print("==================================================================\n")

def correr_servidor():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

hilo = threading.Thread(target=correr_servidor, daemon=True)
hilo.start()
time.sleep(2)


🚀 CONFIGURACIÓN DE BOT DE COCINA EXITOSA 🚀
Configura esta URL exacta como tu Callback URL en Meta Dashboard:
https://mutate-stylus-facedown.ngrok-free.dev/webhook



In [19]:
# Mandar el primer mensaje para iniciar el flujo con wats y verificar que este online.

payload_1 = {
    "entry": [{"changes": [{"value": {"messages": [
        {"from": "5215614857733", "text": {"body": "Hola"}} #
    ]}}]}]
}

r1 = requests.post("http://127.0.0.1:8000/webhook", json=payload_1)
print("Mensaje 1:", r1.status_code, r1.text)

Mensaje 1: 200 {"status":"ok"}
